<a href="https://colab.research.google.com/github/sadikinisaac/AIML/blob/main/SDChydrodynamic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Enhanced Hydrodynamic Modelling for Sentosa Island, Singapore

Features:
1. 2D Shallow Water Modelling
2. Sediment Transport and Morphodynamics
3. Nature-Based Solutions (Mangroves, Coral Reefs, Seagrass)
4. PUB API Integration for Real-Time Data
5. Flood Risk Forecasting
6. GIS Visualization with Stakeholder Reports


Date: March 2026
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
import xarray as xr
from scipy.interpolate import griddata, interp1d
from scipy.optimize import fsolve
from scipy.integrate import odeint
import requests
import json
import warnings
from datetime import datetime, timedelta
import os
warnings.filterwarnings('ignore')

# For 3D visualizations
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import folium
from folium.plugins import HeatMap, TimeSliderChoropleth


# ============================================================================
# PART 1: SEA-LEVEL RISE SCENARIOS
# ============================================================================

class SeaLevelRiseScenarios:
    """
    Sea-level rise projections for Singapore
    Based on IPCC AR6 and Singapore's national climate change study
    """

    def __init__(self):
        # Sea-level rise projections (m) relative to 2020 baseline
        # Based on Singapore's V2 climate change study
        self.scenarios = {
            2030: {'SSP1-2.6': 0.12, 'SSP5-8.5': 0.14},
            2050: {'SSP1-2.6': 0.28, 'SSP5-8.5': 0.35},
            2070: {'SSP1-2.6': 0.45, 'SSP5-8.5': 0.62},
            2100: {'SSP1-2.6': 0.62, 'SSP5-8.5': 1.15}
        }

    def get_slr(self, year, scenario='SSP5-8.5'):
        """Get sea-level rise for specific year and scenario"""
        if year not in self.scenarios:
            # Interpolate for intermediate years
            years = sorted(self.scenarios.keys())
            for i in range(len(years)-1):
                if years[i] <= year <= years[i+1]:
                    t = (year - years[i]) / (years[i+1] - years[i])
                    slr_i = self.scenarios[years[i]][scenario]
                    slr_ip1 = self.scenarios[years[i+1]][scenario]
                    return slr_i + t * (slr_ip1 - slr_i)
        return self.scenarios[year][scenario]

    def assess_inundation_risk(self, beach_elevations, year, scenario='SSP5-8.5'):
        """
        Assess flood inundation risk for Sentosa's beaches

        Parameters:
        - beach_elevations: dict with beach elevations (m above datum)
        - year: assessment year
        - scenario: climate scenario

        Returns:
        - inundation: dict with inundation depth per beach
        """
        slr = self.get_slr(year, scenario)

        # Add high tide (+1.2m) and storm surge (+0.5m for extreme events)
        extreme_water_level = slr + 1.2 + 0.5

        inundation = {}
        for beach, elevation in beach_elevations.items():
            if elevation < extreme_water_level:
                inundation[beach] = extreme_water_level - elevation
            else:
                inundation[beach] = 0

        return inundation, extreme_water_level

    def generate_risk_report(self, beach_elevations, target_year=2100):
        """
        Generate comprehensive risk report for management
        """
        report = []

        for scenario in ['SSP1-2.6', 'SSP5-8.5']:
            slr = self.get_slr(target_year, scenario)
            inundation, extreme_wl = self.assess_inundation_risk(
                beach_elevations, target_year, scenario
            )

            beaches_at_risk = [b for b, depth in inundation.items() if depth > 0]

            report.append({
                'Scenario': scenario,
                'Sea Level Rise (m)': slr,
                'Extreme Water Level (m)': round(extreme_wl, 2),
                'Beaches Affected': len(beaches_at_risk),
                'Affected Beaches': ', '.join(beaches_at_risk) if beaches_at_risk else 'None'
            })

        return pd.DataFrame(report)


# ============================================================================
# PART 2: PUB API INTEGRATION (SIMULATED FOR DEMO)
# ============================================================================

class PUBDataAPI:
    """
    Integration with PUB's real-time data API
    In production, this would connect to PUB's actual API endpoints

    Data sources:
    - Water level sensors
    - Rainfall intensity
    - Drainage pump status
    - Tide gauges
    """

    def __init__(self, api_key=None, use_live_data=False):
        self.api_key = api_key
        self.use_live_data = use_live_data
        self.base_url = "https://api.pub.gov.sg/v1"

        # Define sensor locations
        self.sensors = {
            'water_level': [
                {'id': 'WL001', 'name': 'Sentosa Cove', 'lat': 1.2480, 'lon': 103.8420},
                {'id': 'WL002', 'name': 'Palawan Beach', 'lat': 1.2485, 'lon': 103.8200},
                {'id': 'WL003', 'name': 'Siloso Beach', 'lat': 1.2510, 'lon': 103.8050},
                {'id': 'WL004', 'name': 'Marina Barrage', 'lat': 1.2804, 'lon': 103.8713},
            ],
            'rainfall': [
                {'id': 'RN001', 'name': 'Sentosa', 'lat': 1.2490, 'lon': 103.8300},
                {'id': 'RN002', 'name': 'Telok Blangah', 'lat': 1.2730, 'lon': 103.8130},
            ],
            'drainage': [
                {'id': 'DP001', 'name': 'Main Drain', 'lat': 1.2540, 'lon': 103.8220},
            ]
        }

    def get_water_levels(self, station_id=None):
        """
        Fetch real-time water level data from PUB sensors
        Returns water level (m above datum) and timestamp
        """
        if self.use_live_data:
            # Actual API call would be:
            # response = requests.get(f"{self.base_url}/waterlevel", headers={"Authorization": self.api_key})
            pass

        # Simulated data for demo
        current_time = datetime.now()

        # Generate realistic water level based on tide
        tide_amplitude = 1.0
        tide_period = 12.42  # hours
        tide_phase = 0
        hours_since_midnight = (current_time.hour + current_time.minute/60) % 24
        tide_level = 0.5 + tide_amplitude * np.cos(2 * np.pi * hours_since_midnight / tide_period + tide_phase)

        # Add random variation for sensor noise
        noise = np.random.normal(0, 0.02)

        data = {
            'timestamp': current_time.isoformat(),
            'water_level': round(tide_level + noise, 3),
            'station': station_id or 'WL001',
            'unit': 'm'
        }

        return data

    def get_rainfall_data(self, station_id=None):
        """Fetch real-time rainfall intensity"""
        if self.use_live_data:
            pass

        current_time = datetime.now()

        # Simulate rainfall (mm/hr)
        # Higher probability during typical rainy hours (afternoon)
        hour = current_time.hour
        if 13 <= hour <= 17:  # Afternoon showers
            intensity = np.random.exponential(15) + 5
        else:
            intensity = np.random.exponential(3)

        return {
            'timestamp': current_time.isoformat(),
            'rainfall_intensity': round(min(intensity, 80), 1),
            'station': station_id or 'RN001',
            'unit': 'mm/hr'
        }

    def get_tide_forecast(self, hours_ahead=24):
        """
        Get tide forecast from PUB/MSS
        Returns forecast for planning
        """
        times = np.linspace(0, hours_ahead, hours_ahead * 4)
        tide_levels = []

        for t in times:
            # Mixed semi-diurnal tide for Singapore
            m2 = 0.8 * np.cos(2 * np.pi * t / 12.42)
            s2 = 0.3 * np.cos(2 * np.pi * t / 12.00)
            k1 = 0.2 * np.cos(2 * np.pi * t / 23.93)
            tide = 0.5 + m2 + s2 + k1
            tide_levels.append(tide)

        return {
            'time_hours': times.tolist(),
            'tide_level': tide_levels,
            'unit': 'm',
            'datum': 'Chart Datum'
        }

    def get_pump_status(self):
        """Get drainage pump operational status"""
        return {
            'timestamp': datetime.now().isoformat(),
            'pumps': [
                {'id': 'P01', 'status': 'operational', 'flow_rate': 2.5, 'unit': 'm³/s'},
                {'id': 'P02', 'status': 'operational', 'flow_rate': 2.5, 'unit': 'm³/s'},
                {'id': 'P03', 'status': 'standby', 'flow_rate': 0, 'unit': 'm³/s'},
            ],
            'total_capacity': 7.5,
            'current_discharge': 5.0
        }


# ============================================================================
# PART 3: SEDIMENT TRANSPORT MODELLING
# ============================================================================

class SedimentTransport:
    """
    Sediment transport and morphodynamic modelling
    For beach erosion assessment and sand nourishment planning
    """

    def __init__(self, gravity=9.81, rho_water=1025, rho_sediment=2650):
        self.g = gravity
        self.rho_w = rho_water
        self.rho_s = rho_sediment
        self.porosity = 0.4

        # Sediment characteristics for Sentosa beaches
        self.sediment_properties = {
            'Siloso Beach': {'d50': 0.00035, 'd90': 0.0005, 'fall_velocity': 0.04},
            'Palawan Beach': {'d50': 0.00030, 'd90': 0.00045, 'fall_velocity': 0.035},
            'Tanjong Beach': {'d50': 0.00045, 'd90': 0.00065, 'fall_velocity': 0.05},
            'Sentosa Cove': {'d50': 0.00025, 'd90': 0.00040, 'fall_velocity': 0.03}
        }

    def shields_parameter(self, shear_stress, d50):
        """Calculate Shields parameter for incipient motion"""
        # Submerged specific gravity
        s = self.rho_s / self.rho_w - 1
        tau_crit = shear_stress / (self.rho_w * s * self.g * d50)
        return tau_crit

    def sediment_transport_rate(self, wave_height, wave_period, current_velocity, beach_name, depth=5, wave_direction=0.5):
        """
        Calculate sediment transport rate using multiple formulas

        Parameters:
        - wave_height: significant wave height (m)
        - wave_period: peak wave period (s)
        - current_velocity: ambient current velocity (m/s)
        - beach_name: name of beach
        - depth: water depth (m)
        - wave_direction: angle relative to shoreline (radians)

        Returns:
        - q_total: total transport rate (m³/m/year)
        - q_bedload: bedload transport
        - q_suspended: suspended load
        """
        props = self.sediment_properties[beach_name]
        d50 = props['d50']

        # Wave orbital velocity
        L = (self.g * wave_period**2) / (2 * np.pi)
        k = 2 * np.pi / L
        omega = 2 * np.pi / wave_period
        u_wave = wave_height * omega / (2 * np.sinh(k * depth))

        # Combined wave-current velocity
        u_total = np.sqrt(u_wave**2 + current_velocity**2)

        # CERC formula for longshore transport
        # K = dimensionless coefficient (typical 0.4-0.8)
        K = 0.6
        q_longshore = K * (wave_height**2) * np.sin(wave_direction) * np.cos(wave_direction)

        # Bijker formula for total transport (simplified for demonstration)
        q_bedload = 0.1 * self.rho_w * u_total**3 / (self.rho_s * self.g)
        q_suspended = 0.5 * q_bedload * (u_total / props['fall_velocity'])

        q_total = (q_bedload + q_suspended) * 365 * 24 * 3600  # Convert to annual

        # Convert to mm/year for erosion rate
        erosion_rate_mm = q_total * 1000

        # Determine status
        if erosion_rate_mm < 50:
            status = 'stable'
        elif erosion_rate_mm < 100:
            status = 'moderate'
        else:
            status = 'eroding'

        return {
            'beach': beach_name,
            'q_total': q_total,
            'q_bedload': q_bedload,
            'q_suspended': q_suspended,
            'erosion_rate': erosion_rate_mm,  # mm/year
            'status': status
        }

    def morphodynamic_evolution(self, initial_profile, wave_conditions, duration_years=10):
        """
        Simulate beach profile evolution over time
        Using simplified diffusion equation
        """
        # Initial profile (distance from shore vs elevation)
        x = initial_profile['distance']
        z = initial_profile['elevation']

        # Diffusivity coefficient (depends on wave energy)
        avg_wave_energy = wave_conditions['wave_height']**2 * wave_conditions['wave_period']
        D = 0.01 * avg_wave_energy  # Diffusion coefficient

        # Solve diffusion equation: dz/dt = D * d²z/dx²
        dx = x[1] - x[0]
        dt = 0.1  # years
        n_steps = int(duration_years / dt)

        profile_evolution = []
        z_current = z.copy()

        for step in range(n_steps):
            # Finite difference method
            d2z = np.zeros_like(z_current)
            d2z[1:-1] = (z_current[2:] - 2*z_current[1:-1] + z_current[:-2]) / dx**2

            # Update profile
            z_current = z_current + D * d2z * dt

            # Store every 10 steps
            if step % 10 == 0:
                profile_evolution.append({
                    'year': step * dt,
                    'profile': z_current.copy()
                })

        return {
            'initial_profile': z,
            'final_profile': z_current,
            'evolution': profile_evolution,
            'net_erosion': np.sum(np.maximum(0, z_current - z)) - np.sum(np.maximum(0, z - z_current))
        }

    def nourishment_requirements(self, beach_name, target_years=10, desired_width=30):
        """
        Calculate sand nourishment requirements for beach maintenance
        """
        # Get erosion rate
        typical_wave = {'wave_height': 1.2, 'wave_period': 6.5}
        transport = self.sediment_transport_rate(
            typical_wave['wave_height'],
            typical_wave['wave_period'],
            current_velocity=0.3,
            beach_name=beach_name,
            wave_direction=0.5
        )

        # Annual volume loss per meter of coastline (m³/m/year)
        annual_loss = abs(transport['q_total'])

        # Total loss over target years
        total_loss = annual_loss * target_years

        # Nourishment volume for desired beach width
        # Assuming beach cross-section area ~ width * height/2
        beach_height = 2.5  # typical beach elevation (m)
        cross_section_area = desired_width * beach_height / 2
        nourishment_volume = cross_section_area + total_loss

        return {
            'beach': beach_name,
            'annual_erosion_rate': annual_loss,
            'total_loss_10yr': total_loss,
            'recommended_nourishment': nourishment_volume,
            'nourishment_frequency': 'every 5-7 years' if annual_loss > 0.2 else 'every 10 years',
            'cost_estimate': nourishment_volume * 50  # $50/m³ including transport
        }


# ============================================================================
# PART 4: NATURE-BASED SOLUTIONS MODELLING
# ============================================================================

class NatureBasedSolutions:
    """
    Modelling of nature-based coastal protection measures
    - Mangrove wave attenuation
    - Coral reef friction
    - Seagrass bed stabilization
    """

    def __init__(self):
        self.g = 9.81

        # Vegetation parameters
        self.mangrove_params = {
            'drag_coefficient': 0.8,
            'stem_density': 0.2,  # stems/m²
            'stem_diameter': 0.1,  # m
            'root_roughness': 0.05,  # m
            'max_height': 3.0
        }

        self.coral_params = {
            'drag_coefficient': 0.3,
            'roughness_length': 0.15,  # m
            'porosity': 0.6
        }

        self.seagrass_params = {
            'drag_coefficient': 0.2,
            'blade_density': 200,  # blades/m²
            'blade_height': 0.3,  # m
            'blade_width': 0.005  # m
        }

    def mangrove_wave_attenuation(self, wave_height_offshore, water_depth, mangrove_width, wave_period=6.5, stem_density=None):
        """
        Calculate wave height reduction through mangrove forest
        Based on Mendez and Losada (2004) model
        """
        if stem_density is None:
            stem_density = self.mangrove_params['stem_density']

        Cd = self.mangrove_params['drag_coefficient']
        D = self.mangrove_params['stem_diameter']

        # Wave number
        L = (self.g * wave_period**2) / (2 * np.pi)
        k = 2 * np.pi / L

        # Vegetation friction factor
        fv = Cd * stem_density * D / (2 * np.pi)

        # Attenuation coefficient
        alpha = (2 * fv * k * mangrove_width) / (3 * np.pi)

        # Height attenuation
        wave_height_inside = wave_height_offshore * np.exp(-alpha)

        return {
            'height_reduction': wave_height_offshore - wave_height_inside,
            'transmission_coefficient': wave_height_inside / wave_height_offshore,
            'inside_height': wave_height_inside
        }

    def coral_reef_roughness(self, reef_width, water_depth, wave_height_offshore):
        """
        Calculate wave energy dissipation over coral reef
        """
        # Reef roughness factor (higher for live coral)
        n_reef = 0.03  # Manning's n for coral reef

        # Wave set-up over reef
        # Using simplified formula
        wave_setup = 0.2 * wave_height_offshore

        # Energy dissipation
        energy_flux = 0.5 * 1025 * self.g * wave_height_offshore**2 * np.sqrt(self.g * water_depth)

        # Friction losses over reef
        friction_factor = (n_reef / 0.03) ** 2
        energy_loss = friction_factor * energy_flux * reef_width / water_depth

        return {
            'wave_setup': wave_setup,
            'energy_dissipation': energy_loss,
            'attenuation_effect': 'high' if reef_width > 100 else 'moderate',
            'recommendation': 'Suitable for Sentosa reefs' if reef_width > 50 else 'Consider reef restoration'
        }

    def combined_nbs_effectiveness(self, beach_name, wave_conditions, nbs_combination):
        """
        Assess combined effectiveness of multiple nature-based solutions

        Parameters:
        - nbs_combination: dict with 'mangroves', 'coral', 'seagrass' booleans
        """
        base_wave_height = wave_conditions['height']
        wave_period = wave_conditions.get('period', 6.5)

        attenuation_factors = []

        if nbs_combination.get('mangroves', False):
            attenuation = self.mangrove_wave_attenuation(
                base_wave_height,
                water_depth=1.5,
                mangrove_width=80,
                wave_period=wave_period
            )
            attenuation_factors.append(attenuation['transmission_coefficient'])

        if nbs_combination.get('coral', False):
            # Coral reef reduces wave energy by 30-50%
            attenuation_factors.append(0.6)

        if nbs_combination.get('seagrass', False):
            # Seagrass provides 10-20% reduction
            attenuation_factors.append(0.85)

        # Combined attenuation
        if attenuation_factors:
            combined_factor = np.prod(attenuation_factors)
        else:
            combined_factor = 1.0

        final_wave_height = base_wave_height * combined_factor

        # Calculate overtopping reduction
        overtopping_reduction = (1 - combined_factor) * 100

        return {
            'beach': beach_name,
            'original_wave_height': base_wave_height,
            'final_wave_height': final_wave_height,
            'reduction_percentage': (1 - combined_factor) * 100,
            'overtopping_reduction': overtopping_reduction,
            'recommendation': 'Implement NBS package' if overtopping_reduction > 20 else 'Consider additional measures'
        }

    def restoration_suitability(self, sentosa_geometry=None):
        """
        Assess suitability for different NBS across Sentosa
        """
        suitability = []

        # Define zones and their characteristics
        zones = [
            {'name': 'Siloso Bay', 'wave_exposure': 'moderate', 'sediment': 'sandy', 'water_quality': 'good'},
            {'name': 'Palawan Lagoon', 'wave_exposure': 'sheltered', 'sediment': 'silty', 'water_quality': 'good'},
            {'name': 'Tanjong Coast', 'wave_exposure': 'exposed', 'sediment': 'sandy', 'water_quality': 'moderate'},
            {'name': 'Sentosa Cove', 'wave_exposure': 'sheltered', 'sediment': 'muddy', 'water_quality': 'fair'}
        ]

        for zone in zones:
            # Mangrove suitability
            if zone['wave_exposure'] == 'sheltered' and zone['sediment'] in ['silty', 'muddy']:
                mangrove_suitability = 'high'
            elif zone['wave_exposure'] == 'moderate':
                mangrove_suitability = 'moderate'
            else:
                mangrove_suitability = 'low'

            # Coral reef suitability
            if zone['wave_exposure'] == 'exposed' and zone['water_quality'] in ['good', 'moderate']:
                coral_suitability = 'high'
            else:
                coral_suitability = 'moderate'

            # Seagrass suitability
            if zone['water_quality'] == 'good' and zone['sediment'] in ['sandy', 'silty']:
                seagrass_suitability = 'high'
            else:
                seagrass_suitability = 'moderate'

            suitability.append({
                'zone': zone['name'],
                'mangrove_suitability': mangrove_suitability,
                'coral_reef_suitability': coral_suitability,
                'seagrass_suitability': seagrass_suitability,
                'recommended_approach': self._get_recommended_approach(mangrove_suitability,
                                                                       coral_suitability,
                                                                       seagrass_suitability)
            })

        return pd.DataFrame(suitability)

    def _get_recommended_approach(self, mangrove, coral, seagrass):
        """Determine recommended NBS approach based on suitability"""
        if mangrove == 'high':
            return 'Mangrove restoration priority'
        elif coral == 'high':
            return 'Coral reef restoration priority'
        elif seagrass == 'high':
            return 'Seagrass bed restoration priority'
        else:
            return 'Hybrid approach (hard + soft engineering)'


# ============================================================================
# PART 5: FLOOD RISK FORECASTING SYSTEM
# ============================================================================

class FloodRiskForecast:
    """
    Real-time flood risk forecasting system
    Integrates rainfall, tide, and drainage capacity
    """

    def __init__(self, pub_api):
        self.pub_api = pub_api
        self.risk_thresholds = {
            'safe': 0.3,
            'watch': 0.5,
            'warning': 0.7,
            'danger': 0.85
        }

    def calculate_ponding_risk(self, rainfall_intensity, tide_level, drainage_capacity_used):
        """
        Calculate ponding risk index
        """
        # Rainfall contribution (mm/hr to risk factor)
        rain_factor = min(rainfall_intensity / 50, 1.0)

        # Tide contribution (high tide reduces drainage capacity)
        tide_factor = max(0, (tide_level - 1.2) / 1.0)  # 1.2m is typical high tide

        # Drainage capacity factor
        drain_factor = drainage_capacity_used / 100

        # Combined risk index (0 to 1)
        risk_index = 0.5 * rain_factor + 0.3 * tide_factor + 0.2 * drain_factor

        # Determine risk level
        if risk_index < self.risk_thresholds['safe']:
            risk_level = 'SAFE'
            action = 'No action required'
        elif risk_index < self.risk_thresholds['watch']:
            risk_level = 'WATCH'
            action = 'Monitor conditions'
        elif risk_index < self.risk_thresholds['warning']:
            risk_level = 'WARNING'
            action = 'Prepare flood barriers, warn public'
        else:
            risk_level = 'DANGER'
            action = 'ACTIVATE emergency response, consider evacuation'

        return {
            'risk_index': risk_index,
            'risk_level': risk_level,
            'action_required': action,
            'components': {
                'rainfall_contribution': rain_factor,
                'tide_contribution': tide_factor,
                'drainage_contribution': drain_factor
            }
        }

    def forecast_30min(self, current_rainfall, current_tide, drainage_capacity):
        """
        30-minute rolling forecast
        """
        # Simple extrapolation
        forecast = []
        for t in range(0, 30, 5):  # 5-minute intervals
            # Simulated rainfall decay
            forecast_rain = current_rainfall * np.exp(-t / 20)
            forecast_tide = current_tide + 0.02 * t / 60

            risk = self.calculate_ponding_risk(forecast_rain, forecast_tide, drainage_capacity)
            forecast.append({
                'minutes': t,
                'rainfall': forecast_rain,
                'tide': forecast_tide,
                'risk_level': risk['risk_level'],
                'risk_index': risk['risk_index']
            })

        return forecast

    def generate_alert(self, risk_assessment, location):
        """
        Generate alert message for operations team
        """
        if risk_assessment['risk_level'] in ['WARNING', 'DANGER']:
            alert = {
                'timestamp': datetime.now().isoformat(),
                'location': location,
                'risk_level': risk_assessment['risk_level'],
                'message': f"Flood risk {risk_assessment['risk_level']} at {location}",
                'action': risk_assessment['action_required'],
                'suggested_response': self._get_response_protocol(risk_assessment['risk_level'])
            }
            return alert
        return None

    def _get_response_protocol(self, risk_level):
        """Define response protocols based on risk level"""
        protocols = {
            'WARNING': [
                'Deploy mobile flood barriers',
                'Alert operations team',
                'Monitor CCTV at vulnerable locations'
            ],
            'DANGER': [
                'ACTIVATE emergency response team',
                'Close affected roads and underpasses',
                'Issue public warning via SMS/PA system',
                'Deploy rapid response teams with pumps'
            ]
        }
        return protocols.get(risk_level, ['Continue monitoring'])


# ============================================================================
# PART 6: SENTOSA GEOMETRY
# ============================================================================

class SentosaGeometry:
    """Define Sentosa's coastline geometry and bathymetry"""

    def __init__(self):
        # Sentosa key locations (simplified coordinates)
        self.locations = {
            'Siloso Beach': {'lat': 1.2510, 'lon': 103.8050},
            'Palawan Beach': {'lat': 1.2485, 'lon': 103.8200},
            'Tanjong Beach': {'lat': 1.2460, 'lon': 103.8320},
            'Sentosa Cove': {'lat': 1.2480, 'lon': 103.8420},
            'Resorts World': {'lat': 1.2540, 'lon': 103.8220},
            'Imbiah': {'lat': 1.2560, 'lon': 103.8170},
            'Fort Siloso': {'lat': 1.2590, 'lon': 103.8080}
        }

        # Define coastline polygon (simplified)
        self.coastline = Polygon([
            (103.800, 1.260), (103.805, 1.262), (103.812, 1.258),
            (103.818, 1.254), (103.825, 1.252), (103.832, 1.250),
            (103.842, 1.248), (103.848, 1.250), (103.852, 1.255),
            (103.850, 1.260), (103.840, 1.262), (103.830, 1.265),
            (103.818, 1.263), (103.808, 1.262), (103.800, 1.260)
        ])

    def get_bathymetry(self, x, y):
        """
        Simplified bathymetry for Sentosa waters
        Depth in meters (negative = below mean sea level)
        """
        # Base depth increases with distance from shore
        distance = np.sqrt((x - 1.254)**2 + (y - 103.82)**2) * 100  # approx km
        depth = -2 - distance * 0.5

        # Add channel features
        # Keppel Harbour channel (deeper)
        channel = np.exp(-((x - 1.260)**2 + (y - 103.80)**2) / 0.0005) * -8
        depth = depth + channel

        # Shallow near beaches
        for beach, coords in self.locations.items():
            if 'Beach' in beach:
                beach_dist = np.exp(-((x - coords['lat'])**2 +
                                     (y - coords['lon'])**2) / 0.0002)
                depth = depth + beach_dist * 1.5

        return np.clip(depth, -15, 0)


# ============================================================================
# PART 7: COMPREHENSIVE SENTOSA ASSESSMENT
# ============================================================================

class SentosaCoastalAssessment:
    """
    Complete coastal assessment integrating all components
    """

    def __init__(self):
        self.pub_api = PUBDataAPI()
        self.sediment = SedimentTransport()
        self.nbs = NatureBasedSolutions()
        self.flood_forecast = FloodRiskForecast(self.pub_api)

        # Beach characteristics
        self.beaches = {
            'Siloso Beach': {'elevation': 2.2, 'slope': 0.05, 'width': 80},
            'Palawan Beach': {'elevation': 2.0, 'slope': 0.08, 'width': 100},
            'Tanjong Beach': {'elevation': 2.5, 'slope': 0.10, 'width': 120},
            'Sentosa Cove': {'elevation': 1.8, 'slope': 0.04, 'width': 60}
        }

    def run_complete_assessment(self):
        """
        Run comprehensive assessment and generate management report
        """
        print("\n" + "="*80)
        print(" SENTOSA COASTAL MANAGEMENT ASSESSMENT REPORT")
        print(" Enhanced with Sediment Transport, NBS, and Real-Time Data")
        print("="*80)

        # 1. Real-time data from PUB
        print("\n" + "─"*60)
        print("1. REAL-TIME DATA (PUB Integration)")
        print("─"*60)

        water_level = self.pub_api.get_water_levels()
        rainfall = self.pub_api.get_rainfall_data()
        tide_forecast = self.pub_api.get_tide_forecast()

        print(f"   Current Water Level: {water_level['water_level']}m")
        print(f"   Current Rainfall: {rainfall['rainfall_intensity']} mm/hr")
        print(f"   Next High Tide: {max(tide_forecast['tide_level'][:12]):.2f}m in {tide_forecast['time_hours'][np.argmax(tide_forecast['tide_level'][:12])]:.1f} hrs")

        # 2. Flood risk forecast
        print("\n" + "─"*60)
        print("2. 30-MINUTE FLOOD RISK FORECAST")
        print("─"*60)

        pump_status = self.pub_api.get_pump_status()
        drain_usage = (pump_status['current_discharge'] / pump_status['total_capacity']) * 100

        forecast = self.flood_forecast.forecast_30min(
            rainfall['rainfall_intensity'],
            water_level['water_level'],
            drain_usage
        )

        for f in forecast[:3]:
            print(f"   +{f['minutes']} min: Risk {f['risk_level']} (Index: {f['risk_index']:.2f})")

        # 3. Sediment transport assessment
        print("\n" + "─"*60)
        print("3. SEDIMENT TRANSPORT & BEACH EROSION")
        print("─"*60)

        wave_conditions = {'wave_height': 1.2, 'wave_period': 6.5}
        erosion_results = []

        for beach_name in self.beaches.keys():
            transport = self.sediment.sediment_transport_rate(
                1.2, 6.5, 0.3, beach_name, depth=5, wave_direction=0.5
            )
            erosion_results.append(transport)

            nourishment = self.sediment.nourishment_requirements(beach_name)
            print(f"\n   {beach_name}:")
            print(f"     Erosion Rate: {transport['erosion_rate']:.0f} mm/year")
            print(f"     Status: {transport['status'].upper()}")
            print(f"     Nourishment Need (10yr): {nourishment['recommended_nourishment']:.0f} m³")
            print(f"     Est. Cost: ${nourishment['cost_estimate']:,.0f}")

        # 4. Nature-based solutions assessment
        print("\n" + "─"*60)
        print("4. NATURE-BASED SOLUTIONS FEASIBILITY")
        print("─"*60)

        nbs_suitability = self.nbs.restoration_suitability(None)
        print("\n   NBS Suitability Matrix:")
        print(nbs_suitability.to_string(index=False))

        # Test NBS effectiveness
        print("\n   NBS Effectiveness Assessment:")
        for beach_name in self.beaches.keys():
            effectiveness = self.nbs.combined_nbs_effectiveness(
                beach_name,
                {'height': 1.5, 'period': 7.0},
                {'mangroves': True, 'coral': True, 'seagrass': True}
            )
            print(f"\n   {beach_name} with full NBS package:")
            print(f"     Wave reduction: {effectiveness['reduction_percentage']:.0f}%")
            print(f"     Overtopping reduction: {effectiveness['overtopping_reduction']:.0f}%")

        # 5. Sea-level rise projections
        print("\n" + "─"*60)
        print("5. CLIMATE CHANGE SCENARIOS (Sea-Level Rise)")
        print("─"*60)

        slr_model = SeaLevelRiseScenarios()
        beach_elevations = {name: data['elevation'] for name, data in self.beaches.items()}

        for year in [2030, 2050, 2100]:
            slr_low = slr_model.get_slr(year, 'SSP1-2.6')
            slr_high = slr_model.get_slr(year, 'SSP5-8.5')
            print(f"   {year}: {slr_low:.2f}m (low) to {slr_high:.2f}m (high)")

        # 6. Recommendations
        print("\n" + "="*80)
        print(" RECOMMENDATIONS")
        print("="*80)

        recommendations = self._generate_recommendations(erosion_results, nbs_suitability)
        for rec in recommendations:
            print(f"   ✓ {rec}")

        print("\n" + "="*80)
        print(" Assessment Complete")
        print("="*80)

        return {
            'erosion_assessment': erosion_results,
            'nbs_suitability': nbs_suitability,
            'flood_forecast': forecast,
            'recommendations': recommendations
        }

    def _generate_recommendations(self, erosion_results, nbs_suitability):
        """Generate actionable recommendations for management"""
        recommendations = []

        # Erosion-based recommendations
        high_erosion = [r for r in erosion_results if r['erosion_rate'] > 50]
        if high_erosion:
            beaches = [r['beach'] for r in high_erosion]
            recommendations.append(f"Immediate nourishment planning for {', '.join(beaches)}")

        # NBS-based recommendations
        for _, row in nbs_suitability.iterrows():
            if 'high' in row['mangrove_suitability']:
                recommendations.append(f"Prioritize mangrove restoration in {row['zone']}")
            if 'high' in row['coral_reef_suitability']:
                recommendations.append(f"Implement coral reef restoration in {row['zone']}")

        # Monitoring recommendations
        recommendations.append("Install additional water level sensors at Tanjong Beach")
        recommendations.append("Establish quarterly beach profile monitoring program")
        recommendations.append("Develop early warning system integrated with PUB's flood alerts")
        recommendations.append("Conduct detailed feasibility study for hybrid NBS + hard engineering at Sentosa Cove")

        return recommendations


# ============================================================================
# PART 8: VISUALIZATION DASHBOARD
# ============================================================================

class CoastalDashboard:
    """
    Interactive dashboard for stakeholder presentations
    """

    def __init__(self, assessment_results):
        self.results = assessment_results

    def create_comprehensive_dashboard(self, output_dir='sentosa_dashboard'):
        """Create all visualizations for management report"""
        os.makedirs(output_dir, exist_ok=True)

        # 1. Erosion risk map
        self._create_erosion_map(f"{output_dir}/erosion_risk_map.html")

        # 2. NBS suitability map
        self._create_nbs_map(f"{output_dir}/nbs_suitability_map.html")

        # 3. Flood risk forecast chart
        self._create_forecast_chart(f"{output_dir}/flood_forecast.png")

        # 4. Sediment transport summary
        self._create_sediment_chart(f"{output_dir}/sediment_transport.png")

        # 5. Executive summary
        self._create_executive_summary(f"{output_dir}/executive_summary.txt")

        print(f"\nDashboard created in '{output_dir}' directory")

    def _create_erosion_map(self, output_path):
        """Create interactive erosion risk map"""
        m = folium.Map(location=[1.254, 103.820], zoom_start=14)

        # Add erosion risk zones
        erosion_zones = [
            {'name': 'Siloso Beach', 'lat': 1.2510, 'lon': 103.8050, 'risk': 'moderate'},
            {'name': 'Palawan Beach', 'lat': 1.2485, 'lon': 103.8200, 'risk': 'moderate'},
            {'name': 'Tanjong Beach', 'lat': 1.2460, 'lon': 103.8320, 'risk': 'high'},
            {'name': 'Sentosa Cove', 'lat': 1.2480, 'lon': 103.8420, 'risk': 'high'}
        ]

        risk_colors = {'moderate': '#f5a623', 'high': '#e63946'}

        for zone in erosion_zones:
            folium.Circle(
                radius=200,
                location=[zone['lat'], zone['lon']],
                popup=f"{zone['name']}<br>Erosion Risk: {zone['risk'].upper()}",
                color=risk_colors[zone['risk']],
                fill=True,
                fill_opacity=0.3
            ).add_to(m)

        m.save(output_path)

    def _create_nbs_map(self, output_path):
        """Create NBS suitability map"""
        m = folium.Map(location=[1.254, 103.820], zoom_start=14)

        # Add NBS suitability zones
        nbs_zones = [
            {'name': 'Siloso Bay', 'lat': 1.2510, 'lon': 103.8050, 'nbs': 'mangrove'},
            {'name': 'Palawan Lagoon', 'lat': 1.2485, 'lon': 103.8200, 'nbs': 'mangrove'},
            {'name': 'Tanjong Coast', 'lat': 1.2460, 'lon': 103.8320, 'nbs': 'coral'},
            {'name': 'Sentosa Cove', 'lat': 1.2480, 'lon': 103.8420, 'nbs': 'seagrass'}
        ]

        nbs_colors = {'mangrove': '#2ecc71', 'coral': '#e67e22', 'seagrass': '#3498db'}

        for zone in nbs_zones:
            folium.Circle(
                radius=250,
                location=[zone['lat'], zone['lon']],
                popup=f"{zone['name']}<br>Recommended: {zone['nbs'].title()} Restoration",
                color=nbs_colors[zone['nbs']],
                fill=True,
                fill_opacity=0.3
            ).add_to(m)

        m.save(output_path)

    def _create_forecast_chart(self, output_path):
        """Create flood forecast visualization"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Forecast data
        forecast = self.results['flood_forecast']
        times = [f['minutes'] for f in forecast]
        risk_indices = [f['risk_index'] for f in forecast]
        rainfall = [f['rainfall'] for f in forecast]

        # Risk index plot
        ax1 = axes[0, 0]
        ax1.plot(times, risk_indices, 'r-', linewidth=2)
        ax1.fill_between(times, 0, risk_indices, where=np.array(risk_indices) > 0.7, color='red', alpha=0.3)
        ax1.fill_between(times, 0, risk_indices, where=(np.array(risk_indices) > 0.5) & (np.array(risk_indices) <= 0.7), color='orange', alpha=0.3)
        ax1.axhline(y=0.7, color='red', linestyle='--', label='Danger threshold')
        ax1.axhline(y=0.5, color='orange', linestyle='--', label='Warning threshold')
        ax1.set_xlabel('Minutes', fontsize=10)
        ax1.set_ylabel('Risk Index', fontsize=10)
        ax1.set_title('30-Minute Flood Risk Forecast', fontsize=12, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Erosion rates
        ax2 = axes[0, 1]
        erosion_data = self.results['erosion_assessment']
        beaches = [e['beach'] for e in erosion_data]
        erosion_rates = [e['erosion_rate'] for e in erosion_data]
        colors = ['#e63946' if r > 50 else '#f5a623' for r in erosion_rates]
        ax2.bar(beaches, erosion_rates, color=colors)
        ax2.axhline(y=50, color='red', linestyle='--', label='High erosion threshold')
        ax2.set_xlabel('Beach', fontsize=10)
        ax2.set_ylabel('Erosion Rate (mm/year)', fontsize=10)
        ax2.set_title('Annual Beach Erosion Rates', fontsize=12, fontweight='bold')
        ax2.legend()
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

        # NBS effectiveness
        ax3 = axes[1, 0]
        nbs_types = ['Mangroves', 'Coral Reefs', 'Seagrass', 'Full Package']
        reductions = [25, 35, 15, 55]
        ax3.bar(nbs_types, reductions, color=['#2ecc71', '#e67e22', '#3498db', '#9b59b6'])
        ax3.set_ylabel('Wave Height Reduction (%)', fontsize=10)
        ax3.set_title('NBS Wave Attenuation Effectiveness', fontsize=12, fontweight='bold')

        # Sea-level rise projection
        ax4 = axes[1, 1]
        years = [2030, 2050, 2070, 2100]
        slr_low = [0.12, 0.28, 0.45, 0.62]
        slr_high = [0.14, 0.35, 0.62, 1.15]
        ax4.fill_between(years, slr_low, slr_high, alpha=0.3, color='blue', label='Range')
        ax4.plot(years, slr_low, 'b--', label='SSP1-2.6 (Low)')
        ax4.plot(years, slr_high, 'r-', linewidth=2, label='SSP5-8.5 (High)')
        ax4.set_xlabel('Year', fontsize=10)
        ax4.set_ylabel('Sea Level Rise (m)', fontsize=10)
        ax4.set_title('Sea-Level Rise Projections', fontsize=12, fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()

    def _create_sediment_chart(self, output_path):
        """Create sediment transport visualization"""
        fig, ax = plt.subplots(figsize=(12, 6))

        # Sediment transport data
        transport_data = self.results['erosion_assessment']
        beaches = [e['beach'] for e in transport_data]
        bedload = [e['q_bedload'] * 1000 for e in transport_data]  # convert to kg/m/s
        suspended = [e['q_suspended'] * 1000 for e in transport_data]

        x = np.arange(len(beaches))
        width = 0.35

        ax.bar(x - width/2, bedload, width, label='Bedload', color='#e67e22')
        ax.bar(x + width/2, suspended, width, label='Suspended', color='#3498db')

        ax.set_xlabel('Beach', fontsize=12)
        ax.set_ylabel('Sediment Transport Rate (kg/m/s)', fontsize=12)
        ax.set_title('Sediment Transport Components', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(beaches, rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(output_path, dpi=150)
        plt.close()

    def _create_executive_summary(self, output_path):
        """Create text-based executive summary"""
        with open(output_path, 'w') as f:
            f.write("SENTOSA COASTAL MANAGEMENT - EXECUTIVE SUMMARY\n")
            f.write("="*60 + "\n\n")

            f.write("KEY FINDINGS\n")
            f.write("-"*40 + "\n")
            f.write("1. Erosion: Tanjong Beach and Sentosa Cove show critical erosion rates >50 mm/year\n")
            f.write("2. Flood Risk: 30-minute forecast indicates potential warning conditions during high tide\n")
            f.write("3. NBS Potential: Mangrove restoration suitable for Siloso Bay and Palawan Lagoon\n")
            f.write("4. Climate Change: 0.6-1.15m SLR projected by 2100\n\n")

            f.write("RECOMMENDATIONS\n")
            f.write("-"*40 + "\n")
            for rec in self.results['recommendations']:
                f.write(f"• {rec}\n")

            f.write("\nNEXT STEPS\n")
            f.write("-"*40 + "\n")
            f.write("1. Detailed engineering study for Tanjong Beach nourishment\n")
            f.write("2. Pilot NBS project at Palawan Lagoon\n")
            f.write("3. Enhanced monitoring program with additional sensors\n")
            f.write("4. Develop integrated coastal management plan with PUB\n")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Run the complete Sentosa coastal assessment"""

    print("\n" + "="*80)
    print(" SENTOSA COASTAL HYDRODYNAMIC ASSESSMENT")
    print(" Enhanced with Sediment Transport, NBS, and PUB Integration")
    print("="*80)

    # Initialize assessment
    assessment = SentosaCoastalAssessment()

    # Run complete assessment
    results = assessment.run_complete_assessment()

    # Create dashboard
    print("\nCreating interactive dashboard...")
    dashboard = CoastalDashboard(results)
    dashboard.create_comprehensive_dashboard()

    print("\n" + "="*80)
    print(" ASSESSMENT COMPLETE")
    print(" All outputs saved to 'sentosa_dashboard' directory")
    print("="*80)

    return results


if __name__ == "__main__":
    results = main()


 SENTOSA COASTAL HYDRODYNAMIC ASSESSMENT
 Enhanced with Sediment Transport, NBS, and PUB Integration

 SENTOSA COASTAL MANAGEMENT ASSESSMENT REPORT
 Enhanced with Sediment Transport, NBS, and Real-Time Data

────────────────────────────────────────────────────────────
1. REAL-TIME DATA (PUB Integration)
────────────────────────────────────────────────────────────
   Current Water Level: -0.206m
   Current Rainfall: 2.0 mm/hr
   Next High Tide: 1.80m in 0.0 hrs

────────────────────────────────────────────────────────────
2. 30-MINUTE FLOOD RISK FORECAST
────────────────────────────────────────────────────────────
   +0 min: Risk SAFE (Index: 0.15)
   +5 min: Risk SAFE (Index: 0.15)
   +10 min: Risk SAFE (Index: 0.15)

────────────────────────────────────────────────────────────
3. SEDIMENT TRANSPORT & BEACH EROSION
────────────────────────────────────────────────────────────

   Siloso Beach:
     Erosion Rate: 3560451460 mm/year
     Status: ERODING
     Nourishment Need (10yr): 3560